# EDA on silver layer data to identify exactly what should be cleaned

In [0]:
%python
from pyspark.sql.functions import col, when, trim, regexp_replace, dense_rank, sum as _sum
from pyspark.sql.window import Window

In [0]:
df_bronze = spark.sql("SELECT * FROM marathos.bronze.raw_marathon_data")

In [0]:
df_bronze.printSchema()
df_bronze.limit(10).display()

In [0]:
df_bronze.groupBy("event_distance_length").count().orderBy(col("count").desc()).display()

In [0]:
df_bronze.filter(~col("event_name").rlike(r"\([A-Z]{3}\)$")).select("event_name").distinct().limit(20).display()

In [0]:
df_days = df_bronze.filter(col("athlete_performance").rlike("(?i)d"))
print(f"Number of rows that contains days (d): {df_days.count()}")

In [0]:
df_days.select("event_name", "event_distance_length", "athlete_performance").limit(20).display()

In [0]:
df_bronze.select(
    _sum(col("athlete_performance").isNull().cast("int")).alias("nulls_performance"),
    _sum(col("athlete_year_of_birth").isNull().cast("int")).alias("nulls_birth_year"),
    _sum(col("athlete_gender").isNull().cast("int")).alias("nulls_gender"),
    _sum(col("athlete_average_speed").isNull().cast("int")).alias("nulls_speed")
).display()